# Traitement et formattage des données

**Projet :** Prédiction de la qualité de soudures sur des aciers

**Données :** *Weld Metal Database* — University of Cambridge, Phase Transformations Group ([lien](https://www.phasetrans.msm.cam.ac.uk/map/data/materials/welddb-b.html))


### Objectif de ce notebook

Ce notebook constitue la **première étape** du projet. Il transforme les fichiers bruts de la base de soudures en un jeu de données propre et exploitable pour l'analyse descriptive, l'ACP puis la modélisation.

Les étapes réalisées sont les suivantes :

1. **Chargement des métadonnées** (`welddb.info`) : extraction du nom, de l'unité et du rôle de chaque variable.
2. **Chargement des données** (`welddb.data`) et nommage des colonnes.
3. **Conversion des valeurs mal formatées** (seuils de détection, intervalles, annotations textuelles) en valeurs numériques.
4. **Retranspcription des ID de soudure** (mis sous forme XX-XXX-XX)
5. **Harmonisation des unités** (ppm → % massique).
6. **Export** des données traitées au format `.csv`.

Chaque choix de pré-traitement est explicité et justifié dans la cellule correspondante.

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import re

In [2]:
# Définition des chemins d'accès au données à traiter, et du chemin d'enregistrement pour les données traitées

path_raw_data = r"..\1. Données\01_raw"
path_data = path_raw_data + r"\welddb.data"
path_data_hearder = path_raw_data + r"\welddb.info"

path_data_processed = r"..\1. Données\02_intermediate"

---
## 1. Lecture des métadonnées

Le fichier `welddb.data` ne contient **pas d'en-tête**. La description des 44 colonnes se trouve dans `welddb.info`, sous la forme :

```
Column 1
Carbon concentration / weight % ; ...
```

On extrait de chaque description :
- le **nom** de la variable (partie avant le `/`) ;
- son **unité** (partie après le `/`, vide si la variable n'a pas d'unité).

In [3]:
# Lecture des infos sur la base de données 
with open (path_data_hearder, 'r') as f:
    text = f.read()

# Extraction des noms de features et des unités associées
lines = [l.strip() for l in text.splitlines() if l.strip]

names, units = [], []

for i, line in enumerate(lines) : 
    if line.startswith("Column") : 
        description = lines[i+1]
        description = description.split(";")[0]

        if "/" in description :
            nom, unite = description.split('/', 1)

        else : 
            nom, unite = description , ""

        names.append(nom.strip())
        units.append(unite.strip())

print(f"{len(names)} variables décrites dans le fichier d'information.")

44 variables décrites dans le fichier d'information.


### 1.1 Rôle de chaque variable

Pour structurer la suite de l'étude, chaque variable est rattachée à une **catégorie physique**, d'après l'ordre des colonnes dans la documentation :

| Colonnes | Rôle | Exemples |
|:---:|---|---|
| 1 – 21 | **Composition chimique** | C, Si, Mn, S, P, Ni, Cr, O, N… |
| 22 – 28 | **Procédé de soudage** | courant, tension, AC/DC, apport de chaleur… |
| 29 – 30 | **Traitement thermique** post-soudage | température, durée |
| 31 – 38 | **Indicateurs de qualité** (cibles potentielles) | limite d'élasticité, résistance, allongement, énergie Charpy… |
| 31 – 38 | **Condition d'essai** | température de l'essai Charpy |
| 39 – 43 | **Microstructure** | ferrite aciculaire, martensite… |
| 44 | **Identifiant** | Weld ID |

> La température Charpy est isolée des indicateurs de qualité : ce n'est pas une mesure de la qualité de la soudure, mais une **condition de l'essai** (l'énergie absorbée dépend fortement de la température à laquelle l'essai est réalisé).

In [4]:
role = []
# Affectation d'un rôle à chaque feature en fonction de son index dans la liste des features
for i, nom in enumerate(names) :
    if i in range(0,21) : 
        role.append("Molecular concentration")
    elif i in range(21, 28) :
        role.append("Physical process")
    elif i in range(28,30) : 
        role.append('Heat treatment')
    elif i in range(30,38) : 
        role.append("Test condition" if nom == "Charpy temperature" else "Quality indicator")
    elif i in range(38,43) : 
        role.append("Microstructure")
    elif i == 43 : 
        role.append("Id")

### 1.2 Type attendu de chaque variable

Toutes les variables sont numériques, à l'exception de **4 variables qualitatives** :
`AC or DC`, `Electrode positive or negative`, `Type of weld` et `Weld ID`.

On regroupe l'ensemble de ces informations dans une table de correspondance, qui servira de **dictionnaire des variables** pour tout le projet.

In [5]:
# Affectation du types de données à chaque feature 
list_of_string_features = ["AC or DC", "Electrode positive or negative", "Type of weld","Weld ID"]
types_of_feature = ["str" if nom in list_of_string_features else "float64" for nom in names]

# Création d'un dictionnaire de correspondance entre les noms de features et les unités
correspondance_between_names_units = pd.DataFrame({"Feature" : names, "Unit" : units, "Role" : role, "Types de variables" : types_of_feature})

---
## 2. Chargement des données brutes

- Les valeurs sont séparées par des **espaces** (`sep=r"\s+"`).
- Les valeurs manquantes sont codées par la lettre **`N`** dans la base ; elles sont converties en `NaN`.

In [6]:
# Lecture de la base de données et affectation des noms de features aux colonnes du dataframe
df = pd.read_csv(path_data, sep = r"\s+", header = None, names = names, na_values = "N")
df.shape

(1652, 44)

---
## 3. Conversion des valeurs mal formatées

### 3.1 Diagnostic

Certaines colonnes censées être numériques sont lues comme du texte par pandas, car elles contiennent des valeurs non standard. On identifie ces colonnes en comparant le type attendu (table de correspondance) au type réellement lu.

In [7]:
# Récupération de la liste des features qui sont mal fomatés (en terme de data types)
list_wrong_formatted_features = correspondance_between_names_units[
    correspondance_between_names_units['Types de variables']
    != df.dtypes.values]['Feature'].tolist()

def valeurs_non_numeriques(df, colonnes):
    """
Renvoie, pour chaque colonne, la liste des valeurs présentes
mais non convertibles en numérique.

Paramètres
----------
df : DataFrame
colonnes : liste des noms de colonnes à vérifier

Retour
------
dict {nom_colonne: [valeurs non convertibles]}
"""
    fautes = {}
    for nom in colonnes:
        presentes = df[nom].notna()
        non_convertibles = pd.to_numeric(df[nom], errors="coerce").isna()
        fautes[nom] = df.loc[presentes & non_convertibles, nom].unique().tolist()
    return fautes

# Aperçu des formats problématiques rencontrés
def non_numeric_values(s):
    s = s.dropna()
    return s[pd.to_numeric(s, errors="coerce").isna()]

inventaire = {
    c: non_numeric_values(df[c]).value_counts().head(5).to_dict()
    for c in df.columns
    if len(non_numeric_values(df[c])) > 0
}
pd.Series(inventaire, name="valeurs non numériques (top 5)").to_frame()

,valeurs non numériques (top 5)
Sulphur concentration,{'<0.002': 7}
Molybdenum concentration,{'<0.01': 2}
Vanadium concentration,"{'<0.0005': 266, '<0.01': 31, '<5': 9, '<0.005..."
Copper concentration,{'<0.01': 14}
Cobalt concentration,{'<0.01': 21}
Tungsten concentration,{'<0.1': 12}
Titanium concentration,"{'<100': 50, '<0.01': 16, '<5': 2, '<10': 2}"
Nitrogen concentration,"{'66totndres': 11, '67tot33res': 7, '61tot34re..."
Aluminium concentration,"{'<5': 343, '<100': 38, '<0.01': 16, '<50': 6}"
Boron concentration,"{'<5': 395, '<10': 24}"


### 3.2 Règles de conversion

Trois types de formats non numériques apparaissent. Ils sont traités de la façon suivante :

| Format | Exemple | Signification | Traitement retenu | Justification |
|---|---|---|---|---|
| `<x` | `<0.01` | Valeur sous la **limite de détection** de l'appareil de mesure | $x / 2$ | Convention usuelle en chimie analytique : la vraie valeur est comprise entre 0 et $x$, on prend le milieu de l'intervalle plutôt que la borne haute, qui surestimerait systématiquement la teneur. |
| `a-b` | `150-200` | **Intervalle** de valeurs | $(a+b)/2$ | Le milieu de l'intervalle est l'estimation la moins biaisée. |
| nombre + texte | `67tot33res`, `158(Hv30)` | Valeur accompagnée d'une **annotation** (méthode de mesure, détail) | nombre en début de chaîne | Le premier nombre correspond à la grandeur mesurée (ex. teneur totale, dureté) ; l'annotation est une information secondaire. |

Toute valeur qui ne correspond à aucun de ces formats est considérée comme **manquante** (`NaN`).

In [8]:
# Reformatage des features numériques qui sont mal formatées

def FormatWord(word) :
    """
    Convertit une valeur mal formatée en nombre (float).

    Les valeurs déjà numériques sont renvoyées telles quelles. Les formats
    spéciaux rencontrés dans la base de soudures sont traités ainsi :

    - "<0.01", "<5"  : limite de détection → on garde le seuil (0.01, 5)
    - "150-200"      : intervalle → on prend la moyenne (175)
    - "67tot33res", "54totndres", "158(Hv30)" : on garde le nombre
      en début de chaîne (67, 54, 158)

    Paramètres
    ----------
    word : str, float ou NaN
        Valeur d'une cellule du DataFrame.

    Retour
    ------
    float
        La valeur convertie, ou NaN si elle est manquante ou non reconnue.

    Exemples
    --------
    >>> FormatWord("0.07")        # 0.07
    >>> FormatWord("<0.01")       # 0.01
    >>> FormatWord("150-200")     # 175.0
    >>> FormatWord("67tot33res")  # 67.0
    """

    if pd.isna(word) : 
        return word

    try : 
        return float(word)
    except ValueError : 
        pass
    
    if word.startswith("<") :
        return float(word.split("<")[1])/2

    elif "-" in word and not word.startswith("-"):
        bas, haut = word.split("-")
        return (float(bas) + float(haut)) / 2

    elif re.match(r"^\d+(\.\d+)?", word):
        return float(re.match(r"^\d+(\.\d+)?", word).group())

In [9]:
# Source bibliographique = début du Weld ID jusqu'à l'année (ex. 'Evans-AlTi-1994-450/1ch1' -> 'Evans-AlTi-1994')
def source_from_id(weld_id):
    m = re.match(r"^(.*?\d{4})", weld_id)
    return m.group(1) if m else re.split(r"[-/ ]", weld_id)[0]

df["Weld ID"] = df["Weld ID"].map(source_from_id)
print(f"{df["Weld ID"].nunique()} sources distinctes")
df["Weld ID"].value_counts().head(10)

148 sources distinctes


Weld ID
Evans-AlTi-1994            114
Evans-StressRelief-1991    110
SvenGret-1990               96
Pat-1981                    79
EvansLetter                 78
Mart                        70
Evans-V/Mn-1991             60
Evans-Nb/Mn-1991            60
Evans-Ti/Mn-1992            60
Evans-Al/CMn-1990           59
Name: count, dtype: int64

### 3.3 Application et vérification

On applique la conversion, puis on **vérifie par une assertion** qu'il ne reste plus aucune valeur non numérique dans les colonnes concernées.

In [10]:
df[list_wrong_formatted_features] = (
    df[list_wrong_formatted_features]
    .map(FormatWord)
    .apply(pd.to_numeric, errors="coerce")
)

# Vérification : plus aucune valeur non convertible
dict_val = valeurs_non_numeriques(df, list_wrong_formatted_features)
assert all(len(v) == 0 for v in dict_val.values()), "Des valeurs non numériques subsistent"

print(f"✔ {len(list_wrong_formatted_features)} colonnes converties en numérique.")

✔ 17 colonnes converties en numérique.


---
## 4. Harmonisation des unités

Les concentrations chimiques ne sont pas toutes exprimées dans la même unité : la plupart sont en **% massique**, mais certains éléments présents à l'état de traces (oxygène, azote, titane, bore…) sont en **ppm massique** (*parts per million by weight*).

Pour pouvoir comparer les teneurs entre elles et les interpréter de façon cohérente, toutes les concentrations sont ramenées en **% massique** :

$$1\ \% \text{ massique} = 10\,000 \text{ ppm} \quad\Longrightarrow\quad x_{\%} = \frac{x_{\text{ppm}}}{10\,000}$$

> Remarque : cette conversion ne change pas le résultat d'une modélisation après normalisation (simple changement d'échelle), mais elle rend les statistiques descriptives directement comparables.

In [11]:
mask_ppm = correspondance_between_names_units["Unit"] == "parts per million by weight"
ppm_features = correspondance_between_names_units.loc[mask_ppm, "Feature"].tolist()

df[ppm_features] = df[ppm_features] / 10_000
correspondance_between_names_units.loc[mask_ppm, "Unit"] = "weight %"

print("Variables converties de ppm en % massique :", ppm_features)

Variables converties de ppm en % massique : ['Oxygen concentration', 'Titanium concentration', 'Nitrogen concentration', 'Aluminium concentration', 'Boron concentration', 'Niobium concentration', 'Tin concentration', 'Arsenic concentration', 'Antimony concentration']


---
## 5. Aperçu du jeu de données traité

In [12]:
print(f"Dimensions : {df.shape[0]} soudures × {df.shape[1]} variables\n")
print(df.dtypes.value_counts().to_string())
df.describe().T

Dimensions : 1652 soudures × 44 variables

float64    40
str         4


,count,mean,std,min,25%,50%,75%,max
Carbon concentration,1652.0,0.075521,0.023898,2.900000e-02,6.175000e-02,0.07400,0.08600,0.1800
Silicon concentration,1652.0,0.328577,0.112455,4.000000e-02,2.700000e-01,0.32000,0.36000,1.1400
Manganese concentration,1652.0,1.202821,0.382137,2.700000e-01,9.400000e-01,1.27000,1.44000,2.2500
Sulphur concentration,1648.0,0.009525,0.011229,1.000000e-03,6.000000e-03,0.00700,0.01000,0.1400
Phosphorus concentration,1642.0,0.012952,0.019627,2.000000e-03,7.000000e-03,0.01000,0.01400,0.2500
Nickel concentration,697.0,0.415034,0.786951,0.000000e+00,0.000000e+00,0.06700,0.26000,3.5000
Chromium concentration,784.0,2.101273,3.026548,0.000000e+00,0.000000e+00,0.53000,2.30000,10.2000
Molybdenum concentration,793.0,0.479159,0.477416,0.000000e+00,0.000000e+00,0.34000,1.01000,1.5000
Vanadium concentration,928.0,0.072889,0.255093,0.000000e+00,2.500000e-04,0.00500,0.04400,2.5000
Copper concentration,578.0,0.172042,0.322995,0.000000e+00,0.000000e+00,0.03000,0.18000,1.6300


---
## 6. Enregistrement

Deux fichiers sont exportés dans `02_intermediate` :
- `welddb_processed.csv` : les données nettoyées ;
- `correspondance_between_names_units.csv` : le dictionnaire des variables (nom, unité, rôle, type).

In [13]:
# Enregistrement des données traitées
df.to_csv(path_data_processed + r"\welddb_processed.csv", index=False, sep = ";")
correspondance_between_names_units.to_csv(path_data_processed + r"\correspondance_between_names_units.csv", index=False, sep = ";")

---
## Synthèse

| Étape | Action | Justification |
|---|---|---|
| Métadonnées | Extraction des noms, unités, rôles et types des 44 variables | Le fichier de données n'a pas d'en-tête ; le rôle structure l'analyse (entrées / cibles) |
| Valeurs manquantes | `N` → `NaN` | Codage spécifique de la base |
| Seuils de détection | `<x` → $x/2$ | Estimation non biaisée d'une valeur comprise entre 0 et $x$ |
| Intervalles | `a-b` → $(a+b)/2$ | Milieu de l'intervalle |
| Annotations | `67tot33res` → 67 | Conservation de la grandeur principale |
| Unités | ppm → % massique | Comparabilité des concentrations |

**Prochaine étape :** analyse descriptive (taux de valeurs manquantes, distributions, corrélations), ACP et choix des variables cibles représentatives de la qualité de soudure.